# jaxctrl Layer 3: which transcription factors must I perturb to control a regulon?

A gene-regulatory network is naturally a **hypergraph**: a TF complex (or a pair of TFs) regulating a target is *one hyperedge*, not a bunch of pairwise edges. jaxctrl's hypergraph-control layer (built on the Liu–Slotine–Barabási / Chen–Surana network-controllability line, which originated in systems biology) lets you ask, on that hypergraph directly:

- **`minimum_driver_nodes`** — the fewest TFs you'd have to actuate to make the network fully controllable;
- **`controllability_profile`** — per-TF: the energy to push the network in that TF's direction, and the Kalman rank when that TF is the sole driver (→ which TFs are easy vs hard to control through);
- **`control_energy`** — the cost of steering the network to a target expression vector under a chosen driver set;
- **`HypergraphControlSystem`** — the linearised dynamics packaged for `is_controllable` / `lqr`, i.e. drug-design on the hypergraph.

We use a small 3-uniform GRN hypergraph (every hyperedge = a TF-pair acting on a target), stylized on *E. coli* global regulators. (`adjacency_tensor` requires a uniform hypergraph; for a real heterogeneous-cardinality network you'd take a uniform decomposition first.)

In [ ]:
import jax
jax.config.update('jax_enable_x64', True)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import jaxctrl
import hgx

## 1. A small GRN as a 3-uniform hypergraph

8 TFs; each hyperedge is `(regulator1, regulator2, target)` — a co-regulating pair plus its target.

In [ ]:
genes = ["crp", "fnr", "arcA", "narL", "ihfA", "fis", "ompR", "soxR"]
gi = {g: i for i, g in enumerate(genes)}
triples = [
    ("crp",  "fis",  "arcA"),   ("fnr",  "arcA", "narL"),   ("ihfA", "fnr",  "ompR"),
    ("crp",  "fnr",  "ompR"),   ("fis",  "ihfA", "crp"),    ("arcA", "narL", "soxR"),
    ("crp",  "arcA", "fnr"),    ("narL", "ompR", "ihfA"),
]
n, m = len(genes), len(triples)
inc = np.zeros((n, m), dtype=np.float32)
for j, (a, b, c) in enumerate(triples):
    for g in (a, b, c):
        inc[gi[g], j] = 1.0
print("incidence:", inc.shape, "| uniform edge size:", set(int(c) for c in inc.sum(0)))
hg = hgx.from_incidence(jnp.asarray(inc))
A3 = jaxctrl.adjacency_tensor(hg, order=3)
print("order-3 adjacency tensor:", A3.shape)

## 2. Minimum driver set

Fewest TFs to actuate for full structural controllability of the (tensor-unfolded) hypergraph dynamics.

In [ ]:
mdn = int(jaxctrl.minimum_driver_nodes(hg))
print(f"minimum_driver_nodes = {mdn}  (out of {n} TFs)")

## 3. Per-TF controllability profile

For each TF as a *sole* driver: control energy to reach a unit perturbation in its direction, and the Kalman rank. Low energy / high rank ⇒ a good handle.

In [ ]:
energies, ranks = jaxctrl.controllability_profile(hg)
order = np.argsort(np.asarray(energies))
print("TF            energy    sole-driver Kalman rank")
for i in order:
    print(f"  {genes[i]:8s}  {float(energies[i]):+8.3f}      {int(ranks[i])}/{n}")
best = [genes[i] for i in order[:3]]
print(f"\neasiest TFs to control through: {best}")

plt.figure(figsize=(7, 4))
o = order[::-1]
plt.barh([genes[i] for i in o], [float(energies[i]) for i in o])
plt.xlabel("control energy to reach a unit perturbation (lower = easier)")
plt.title("Per-TF controllability profile"); plt.tight_layout(); plt.show()

## 4. Control-energy landscape: steering to a target expression state

Minimum control energy (on the tensor-unfolded linear surrogate of the hypergraph dynamics) to drive the network from rest to a target expression vector — here **"soxR-on"** (an oxidative-stress response) — under two different actuated-TF sets. The point: the cost is *highly* sensitive to which TFs you actuate, so the choice of driver set is the design decision.


In [ ]:
xf = jnp.zeros(n).at[gi["soxR"]].set(1.0)

set_A = jnp.asarray([gi["crp"], gi["fnr"]])                     # the global regulators CRP, FNR
set_B = jnp.asarray([int(order[-1]), int(order[-2])])          # the two hardest-to-control TFs (from §3)
e_A = float(jaxctrl.control_energy(hg, driver_nodes=set_A, xf=xf, T=2.0))
e_B = float(jaxctrl.control_energy(hg, driver_nodes=set_B, xf=xf, T=2.0))
print(f"control_energy( actuate {{CRP, FNR}}      -> soxR-on, T=2 ): {e_A:,.0f}")
print(f"control_energy( actuate {{{genes[int(set_B[0])]}, {genes[int(set_B[1])]}}} -> soxR-on, T=2 ): {e_B:,.0f}")
print(f"\nratio = {e_A / e_B:.1f}x  (one-to-two orders of magnitude either way is typical)")
print("\n(control_energy uses the tensor-unfolded *linear* surrogate -- a relative cost landscape over\n driver sets, not a calibrated physical energy; reaching a specific target vector is also a\n different question from the generic controllability profile of §3.)")


## 5. Drug-design on the hypergraph: `HypergraphControlSystem` + LQR

Package the hypergraph dynamics as a linear control system actuated by the CRP+FNR pair, check controllability, and synthesise an LQR feedback law $u = -K\,x$. (Note `minimum_driver_nodes` reported 2 — but that is a *generic* count; a specific pair may land one mode short of full rank, in which case the system is still **stabilizable** and LQR is well posed. Swapping in the `controllability_profile` top-2 typically closes the gap.)


In [ ]:
sys = jaxctrl.HypergraphControlSystem(hg, driver_nodes=set_A)
rank, ok = sys.controllability()
print(f"system: {n} states, {set_A.shape[0]} inputs (CRP, FNR)")
print(f"Kalman rank {int(rank)}/{n}  ->  controllable = {bool(ok)}")
K, X = sys.lqr(jnp.eye(n), jnp.eye(int(set_A.shape[0])))
print(f"LQR gain K: shape {tuple(K.shape)};  cost trace(X) = {float(jnp.trace(X)):.3f}")
print("\nK (rows = CRP, FNR inputs; cols = TFs):")
print(np.asarray(K).round(3))

---

**Scaling up.** Swap `inc` for a real topology — **RegulonDB** (*E. coli* TF→gene; σ-factors and TF complexes become hyperedges) or **YEASTRACT** (*S. cerevisiae*) — after a uniform decomposition, and the same four calls give you the minimum driver-gene set, the per-gene controllability profile, the control-energy landscape, and an LQR design. This is the direct biological instantiation of the network-controllability results behind jaxctrl's Layer 3 (Liu, Slotine & Barabási, *Nature* 2011; Chen & Surana, *IEEE TNSE* 2021).